### Transformer Demo (1 sample pair, 1 update) , (N=1, d_model = 4, h = 2, d_ff = 8)

In [96]:
import math
import torch
import torch.nn as nn
from transformers import AutoTokenizer
import torch.nn.functional as F

device = torch.accelerator.current_accelerator().type if torch.accelerator.is_available else "cpu"
print(f"Using {device} device")



# device = torch.device(
#     "cuda" if torch.cuda.is_available()
#     else "cpu"
# )


# print(f"Using {device} device")




Using cuda device


In [97]:
d_model = 4
h = 2
N = 1
d_ff = 8
dropout_p = 0.1
max_length = 128

# tokenizer  = AutoTokenizer.from_pretrained(
#     "google-bert/bert-base-multilingual-cased"
# )

# tokenizer.add_special_tokens({
#     "bos_token" : "<BOS>",
#     "eos_token" : "<EOS>"
# })

# vocab_size = 16
# # vocab_size = len(tokenizer)


In [98]:
vocab = [
    "<PAD>",  # 0
    "<BOS>",  # 1
    "<EOS>",  # 2
    "I",      # 3
    "'",      # 4
    "m",      # 5
    "the",    # 6
    "best",   # 7
    "boxer",  # 8
    ".",      # 9
    "나는",    # 10
    "최고",    # 11
    "##의",    # 12
    "복",      # 13
    "##서",    # 14
    "##다",    # 15
]




source_text = "I'm the best boxer."
target_text = "나는 최고의 복서다."



# source_token_ids = tokenizer.encode(
#     source_text,
#     add_special_tokens = False
# )

# target_token_ids = tokenizer.encode(
#     target_text,
#     add_special_tokens = False
# )

# source_input_ids = [
#     *source_token_ids,
#     tokenizer.eos_token_id
# ]

# decoder_input_ids =[
#     tokenizer.bos_token_id,
#     *target_token_ids
# ]



# labels = [
#     *target_token_ids,
#     tokenizer.eos_token_id
# ]


source_tokens = [
    "I", "'", "m", "the", "best", "boxer", "."
]

target_tokens = [
    "나는", "최고", "##의", "복", "##서", "##다", "."
]

token_to_id = {
    token: token_id
    for token_id, token in enumerate(vocab)
}

id_to_token = {
    token_id: token
    for token, token_id in token_to_id.items()
}




vocab_size = len(vocab)  # 16

pad_token_id = token_to_id["<PAD>"]  # 0
bos_token_id = token_to_id["<BOS>"]  # 1
eos_token_id = token_to_id["<EOS>"]  # 2

source_token_ids = [
    token_to_id[token]
    for token in source_tokens
]

target_token_ids = [
    token_to_id[token]
    for token in target_tokens
]

source_input_ids = [
    *source_token_ids,
    eos_token_id,
]

# Decoder 입력: <BOS> + 번역문
decoder_input_ids = [
    bos_token_id,
    *target_token_ids,
]


source_input_ids = torch.tensor(
    [source_input_ids],
    dtype = torch.long,
    device = device
)

decoder_input_ids = torch.tensor(
    [decoder_input_ids],
    dtype = torch.long,
    device = device
)

# labels = torch.tensor(
#     [labels],
#     dtype = torch.long,
#     device = device
# )


labels = torch.tensor(        
    [[10, 11, 12, 13, 14, 15, 9, 2]],     
    dtype=torch.long,
    device=device,
)


print("Vocab size:", vocab_size)

print("Source text:", source_text)
print("Target text:", target_text)

print("Encoder IDs:", source_input_ids.tolist())
print(
    "Encoder tokens:",
    [id_to_token[token_id]
     for token_id in source_input_ids[0].tolist()]
)

print("Decoder input IDs:", decoder_input_ids.tolist())
print(
    "Decoder input tokens:",
    [id_to_token[token_id]
     for token_id in decoder_input_ids[0].tolist()]
)

print("Label IDs:", labels.tolist())
print(
    "Label tokens:",
    [id_to_token[token_id]
     for token_id in labels[0].tolist()]
)


Vocab size: 16
Source text: I'm the best boxer.
Target text: 나는 최고의 복서다.
Encoder IDs: [[3, 4, 5, 6, 7, 8, 9, 2]]
Encoder tokens: ['I', "'", 'm', 'the', 'best', 'boxer', '.', '<EOS>']
Decoder input IDs: [[1, 10, 11, 12, 13, 14, 15, 9]]
Decoder input tokens: ['<BOS>', '나는', '최고', '##의', '복', '##서', '##다', '.']
Label IDs: [[10, 11, 12, 13, 14, 15, 9, 2]]
Label tokens: ['나는', '최고', '##의', '복', '##서', '##다', '.', '<EOS>']


In [99]:
class SinusoidalPositionalEncoding(nn.Module):
    def __init__(
        self,
        d_model,
        max_length=128,
        dropout_p=0.1
    ):
        super().__init__()

        self.dropout = nn.Dropout(dropout_p)

        position = torch.arange(
            max_length,
            dtype=torch.float32
        ).unsqueeze(1)

        div_term = torch.exp(
            torch.arange(
                0,
                d_model,
                2,
                dtype=torch.float32
            )
            * (-math.log(10000.0) / d_model)
        )

        pe = torch.zeros(
            1,
            max_length,
            d_model
        )

        pe[0, :, 0::2] = torch.sin(
            position * div_term
        )

        odd_size = pe[0, :, 1::2].shape[-1]

        pe[0, :, 1::2] = torch.cos(
            position * div_term[:odd_size]
        )

        self.register_buffer("pe", pe)

    def forward(self, x):
        token_len = x.size(1)

        pe = self.pe[:, :token_len].to(
            dtype=x.dtype
        )

        return self.dropout(x + pe)

In [100]:
import math
import torch
import torch.nn as nn


class TranslationTransformer(nn.Module):
    def __init__(
        self,
        vocab_size,
        pad_token_id,
        d_model=10,
        h=2,
        N=1,
        d_ff=40,
        dropout_p=0.1,
        max_length=128
    ):
        super().__init__()


        self.d_model = d_model
        self.pad_token_id = pad_token_id

        self.embedding = nn.Embedding(
            num_embeddings=vocab_size,
            embedding_dim=d_model,
            padding_idx=pad_token_id
        )

      
        self.positional_encoding = (
            SinusoidalPositionalEncoding(
                d_model=d_model,
                max_length=max_length,
                dropout_p=dropout_p
            )
        )

        

        encoder_layer = nn.TransformerEncoderLayer(
            d_model=d_model,
            nhead=h,
            dim_feedforward=d_ff,
            dropout=dropout_p,
            activation="relu",
            batch_first=True,
            norm_first=False
        )

        self.encoder = nn.TransformerEncoder(
            encoder_layer,
            num_layers=N
        )

  
        decoder_layer = nn.TransformerDecoderLayer(
            d_model=d_model,
            nhead=h,
            dim_feedforward=d_ff,
            dropout=dropout_p,
            activation="relu",
            batch_first=True,
            norm_first=False
        )

        self.decoder = nn.TransformerDecoder(
            decoder_layer,
            num_layers=N
        )

        # Decoder output → vocabulary logits
        self.output_projection = nn.Linear(
            d_model,
            vocab_size,
        )

    def make_causal_mask(
        self,
        target_len,
        device
    ):
       

        return torch.triu(
            torch.ones(
                target_len,
                target_len,
                dtype=torch.bool,
                device=device
            ),
            diagonal=1
        )

    def forward(
        self,
        source_input_ids,
        decoder_input_ids
    ):
      

        source_padding_mask = (
            source_input_ids == self.pad_token_id
        )

        target_padding_mask = (
            decoder_input_ids == self.pad_token_id
        )



        source = self.embedding(
            source_input_ids
        )

        source = (
            source * math.sqrt(self.d_model)
        )

        source = self.positional_encoding(
            source
        )

      

        memory = self.encoder(
            source,
            src_key_padding_mask=source_padding_mask
        )

       
     

        target = self.embedding(
            decoder_input_ids
        )

        target = (
            target * math.sqrt(self.d_model)
        )

        target = self.positional_encoding(
            target
        )

        

        target_len = decoder_input_ids.size(1)

        causal_mask = self.make_causal_mask(
            target_len=target_len,
            device=decoder_input_ids.device
        )

       

        decoder_output = self.decoder(
            tgt=target,

            memory=memory,

            tgt_mask=causal_mask,
         
            tgt_key_padding_mask=target_padding_mask,

            memory_key_padding_mask=source_padding_mask
        )

       

        logits = self.output_projection(
            decoder_output
        )

      

        return logits, memory, decoder_output, source, target

In [101]:
model = TranslationTransformer(
    vocab_size=vocab_size,
    pad_token_id=token_to_id["<PAD>"], 
    d_model=d_model,
    h=h,
    N=N,
    d_ff=d_ff,
    dropout_p=dropout_p,
    max_length=max_length
).to(device)



### using optimizer.step

In [102]:

output_linear_candidates = [
    (name, module)
    for name, module in model.named_modules()
    if (
        isinstance(module, nn.Linear)
        and module.out_features == vocab_size
    )
]

if not output_linear_candidates:
    raise RuntimeError(
        f"out_features={vocab_size}인 Linear 레이어를 찾지 못했습니다."
    )

last_linear_name, last_linear = output_linear_candidates[-1]

if last_linear.bias is None:
    raise RuntimeError("마지막 Linear에 bias가 없습니다.")

print("마지막 Linear:", last_linear_name)
print("W shape:", tuple(last_linear.weight.shape))
print("b shape:", tuple(last_linear.bias.shape))


learning_rate = 1e-3
beta1 = 0.9
beta2 = 0.999
epsilon = 1e-8

optimizer = torch.optim.Adam(
    model.parameters(),
    lr=learning_rate,
    betas=(beta1, beta2),
    eps=epsilon,
    weight_decay=0.0,
)



model.train()
optimizer.zero_grad(set_to_none=True)

model_output, encoder_output, decoder_output, source, target = model(
    source_input_ids,
    decoder_input_ids,
)


print("Encoder output:", encoder_output.shape)
print("Decoder output:", decoder_output.shape)
print("Logits:", model_output.shape)
print("Labels:", labels.shape)

print("\nModel output shape:", tuple(model_output.shape))
print("Labels shape:", tuple(labels.shape))



with torch.no_grad():
    probability_sum = model_output.sum(dim=-1)

    output_is_probability = (
        model_output.min().item() >= 0.0
        and model_output.max().item() <= 1.0
        and torch.allclose(
            probability_sum,
            torch.ones_like(probability_sum),
            atol=1e-4,
            rtol=1e-4,
        )
    )

flat_output = model_output.reshape(-1, vocab_size)
flat_labels = labels.reshape(-1)

if output_is_probability:
    print("모델 출력 형식: probabilities (Softmax 적용됨)")

    loss = F.nll_loss(
        torch.log(flat_output.clamp_min(1e-12)),
        flat_labels,
        ignore_index=pad_token_id,
        reduction="mean",
        label_smoothing=0.1,

    )

else:
    print("모델 출력 형식: raw logits")

    # 권장 방식: Softmax 이전 logits 사용
    loss = F.cross_entropy(
        flat_output,
        flat_labels,
        ignore_index=pad_token_id,
        reduction="mean",
        label_smoothing=0.1,
    )

print("Loss:", loss.item())


# ============================================================
# 5. 업데이트 전 마지막 Linear의 W, b 저장
# ============================================================
W_before = last_linear.weight.detach().clone()
b_before = last_linear.bias.detach().clone()


# ============================================================
# 6. Backward
# ============================================================
loss.backward()

if last_linear.weight.grad is None:
    raise RuntimeError("마지막 Linear의 weight gradient가 없습니다.")

if last_linear.bias.grad is None:
    raise RuntimeError("마지막 Linear의 bias gradient가 없습니다.")


# optimizer.step()에 사용될 gradient 저장
dW = last_linear.weight.grad.detach().clone()
db = last_linear.bias.grad.detach().clone()


# ============================================================
# 7. Gradient 출력
# ============================================================
print("\n" + "=" * 80)
print("마지막 Linear의 Gradient")
print("=" * 80)

print("\ndW shape:", tuple(dW.shape))
print("dW:")
print(dW.cpu())

print("\ndb shape:", tuple(db.shape))
print("db:")
print(db.cpu())


# ============================================================
# 8. Adam optimizer로 모델 전체 파라미터 업데이트
# ============================================================
optimizer.step()


# ============================================================
# 9. 업데이트된 W, b 저장
# ============================================================
W_after = last_linear.weight.detach().clone()
b_after = last_linear.bias.detach().clone()

delta_W = W_after - W_before
delta_b = b_after - b_before


# ============================================================
# 10. 업데이트 전후 파라미터 출력
# ============================================================
print("\n" + "=" * 80)
print("업데이트 전 마지막 Linear 파라미터")
print("=" * 80)

print("\nW_before:")
print(W_before.cpu())

print("\nb_before:")
print(b_before.cpu())


print("\n" + "=" * 80)
print("Adam 업데이트 변화량")
print("=" * 80)

print("\ndelta_W = W_after - W_before:")
print(delta_W.cpu())

print("\ndelta_b = b_after - b_before:")
print(delta_b.cpu())


print("\n" + "=" * 80)
print("업데이트된 마지막 Linear 파라미터")
print("=" * 80)

print("\nW_after:")
print(W_after.cpu())

print("\nb_after:")
print(b_after.cpu())


# ============================================================
# 11. 업데이트 여부 확인
# ============================================================
print("\n" + "=" * 80)
print("업데이트 확인")
print("=" * 80)

print(
    "W가 변경되었는가:",
    not torch.equal(W_before, W_after),
)

print(
    "b가 변경되었는가:",
    not torch.equal(b_before, b_after),
)

print(
    "W 최대 변화량:",
    delta_W.abs().max().item(),
)

print(
    "b 최대 변화량:",
    delta_b.abs().max().item(),
)

print(
    "dW norm:",
    dW.norm().item(),
)

print(
    "db norm:",
    db.norm().item(),
)


# ============================================================
# 12. Adam 내부 상태 확인
# ============================================================
W_adam_state = optimizer.state[last_linear.weight]
b_adam_state = optimizer.state[last_linear.bias]

print("\n" + "=" * 80)
print("Adam 내부 상태")
print("=" * 80)

print("\nW Adam step:", W_adam_state["step"])
print("W 1차 모멘트 m:")
print(W_adam_state["exp_avg"].detach().cpu())

print("\nW 2차 모멘트 v:")
print(W_adam_state["exp_avg_sq"].detach().cpu())

print("\nb Adam step:", b_adam_state["step"])
print("b 1차 모멘트 m:")
print(b_adam_state["exp_avg"].detach().cpu())

print("\nb 2차 모멘트 v:")
print(b_adam_state["exp_avg_sq"].detach().cpu())

마지막 Linear: output_projection
W shape: (16, 4)
b shape: (16,)
Encoder output: torch.Size([1, 8, 4])
Decoder output: torch.Size([1, 8, 4])
Logits: torch.Size([1, 8, 16])
Labels: torch.Size([1, 8])

Model output shape: (1, 8, 16)
Labels shape: (1, 8)
모델 출력 형식: raw logits
Loss: 2.9411728382110596

마지막 Linear의 Gradient

dW shape: (16, 4)
dW:
tensor([[-0.0105,  0.0257, -0.0084, -0.0068],
        [ 0.0139, -0.0180,  0.0018,  0.0023],
        [ 0.0518, -0.1436, -0.0696,  0.1614],
        [-0.0209,  0.0202,  0.0174, -0.0166],
        [-0.0143,  0.0071, -0.0031,  0.0103],
        [-0.0107, -0.0124,  0.0209,  0.0021],
        [-0.0044,  0.0693, -0.0087, -0.0562],
        [ 0.0315, -0.0102, -0.0128, -0.0085],
        [ 0.0115, -0.0345,  0.0137,  0.0093],
        [ 0.1334, -0.1887,  0.1176, -0.0623],
        [ 0.0983,  0.1095, -0.1042, -0.1036],
        [-0.1642, -0.0473,  0.0652,  0.1462],
        [ 0.0569, -0.1502,  0.0988, -0.0055],
        [-0.1393,  0.2226,  0.0029, -0.0862],
        [ 0.0698

In [103]:
Z = model_output
Memory = encoder_output
H = decoder_output


print("source(encoder_input): \n")
print(source,'\n')

print("targeet(decoder_input): \n")
print(target,'\n')

print("Z\n")
print(Z,'\n')
print("Memory\n")
print(Memory,'\n')
print("H\n")
print(H,'\n')




source(encoder_input): 

tensor([[[-1.2988, -0.8680, -0.0000,  2.0705],
         [ 3.1816, -0.0709, -3.7019,  0.2432],
         [ 2.7707,  2.8121,  2.8923,  0.9211],
         [ 0.3370, -3.3615, -1.3232,  2.8886],
         [-0.4345, -0.6972, -3.4385, -0.9628],
         [-3.6826,  1.9672,  0.0000,  2.5295],
         [-0.0000,  0.6220,  1.7887,  1.0234],
         [-2.6495,  0.0000, -0.2531, -3.4356]]], device='cuda:0', grad_fn=<NativeDropoutBackward0>) 

targeet(decoder_input): 

tensor([[[-0.0472,  0.6366,  3.6054,  3.1488],
         [ 0.7744,  1.5849,  0.0959, -1.0266],
         [-1.9268,  1.8311,  1.1024,  1.4398],
         [ 0.2897, -1.6790,  1.1504,  1.9427],
         [-2.2416, -0.0000,  2.2644, -1.3005],
         [ 0.5427, -0.1982,  1.9121,  2.7588],
         [-5.5403,  2.0825,  0.0114,  3.6582],
         [-0.9758,  0.3928,  1.7998,  0.0000]]], device='cuda:0', grad_fn=<NativeDropoutBackward0>) 

Z

tensor([[[ 0.1031, -0.0981, -0.1706,  0.2486,  0.3153,  0.6106, -0.6740, -0.3399,  0

In [104]:
criterion = nn.CrossEntropyLoss(
    ignore_index = token_to_id["<PAD>"],
    label_smoothing = 0.1
)

loss = criterion(
    model_output.reshape(-1, model_output.size(-1)),
    labels.reshape(-1)
)

print("loss:", loss.item())

loss: 2.9411728382110596


In [105]:
P = torch.softmax(Z, dim=-1)
P




tensor([[[0.0513, 0.0419, 0.0390, 0.0593, 0.0634, 0.0852, 0.0236, 0.0329, 0.0756, 0.0733, 0.0676, 0.1074, 0.0271, 0.0220, 0.0342, 0.1961],
         [0.0567, 0.0555, 0.0481, 0.0248, 0.0112, 0.0147, 0.1499, 0.1394, 0.0394, 0.0799, 0.0442, 0.0578, 0.1408, 0.0564, 0.0527, 0.0284],
         [0.1009, 0.0262, 0.0467, 0.0624, 0.0461, 0.0192, 0.1679, 0.0628, 0.0148, 0.0439, 0.0214, 0.0462, 0.1229, 0.1212, 0.0761, 0.0213],
         [0.0404, 0.0717, 0.0556, 0.0165, 0.0224, 0.0298, 0.0275, 0.1007, 0.0890, 0.1811, 0.0607, 0.0888, 0.0478, 0.0157, 0.0423, 0.1102],
         [0.0612, 0.0402, 0.0319, 0.1042, 0.0341, 0.0770, 0.0824, 0.0383, 0.0624, 0.0356, 0.0829, 0.0999, 0.0541, 0.0522, 0.0330, 0.1105],
         [0.0431, 0.0674, 0.0585, 0.0170, 0.0291, 0.0309, 0.0240, 0.0932, 0.0818, 0.1927, 0.0543, 0.0880, 0.0442, 0.0156, 0.0457, 0.1146],
         [0.1094, 0.0293, 0.0578, 0.0642, 0.0886, 0.0283, 0.0931, 0.0570, 0.0193, 0.0678, 0.0232, 0.0587, 0.0881, 0.0897, 0.0868, 0.0386],
         [0.0792, 0.0284, 0

In [106]:


V = 16
epsilon = 0.1

# labels shape: [1, 8]
one_hot_labels = F.one_hot(
    labels,
    num_classes=V,
).to(dtype=model_output.dtype)

Q = (
    (1.0 - epsilon) * one_hot_labels
    + epsilon / V
)

In [107]:
G = (P[0] - Q[0])/target.shape[1]
G

tensor([[ 0.0056,  0.0045,  0.0041,  0.0066,  0.0071,  0.0099,  0.0022,  0.0033,  0.0087,  0.0084, -0.1048,  0.0126,  0.0026,  0.0020,  0.0035,  0.0237],
        [ 0.0063,  0.0062,  0.0052,  0.0023,  0.0006,  0.0011,  0.0180,  0.0166,  0.0041,  0.0092,  0.0047, -0.1061,  0.0168,  0.0063,  0.0058,  0.0028],
        [ 0.0118,  0.0025,  0.0051,  0.0070,  0.0050,  0.0016,  0.0202,  0.0071,  0.0011,  0.0047,  0.0019,  0.0050, -0.0979,  0.0144,  0.0087,  0.0019],
        [ 0.0043,  0.0082,  0.0062,  0.0013,  0.0020,  0.0029,  0.0027,  0.0118,  0.0103,  0.0219,  0.0068,  0.0103,  0.0052, -0.1113,  0.0045,  0.0130],
        [ 0.0069,  0.0042,  0.0032,  0.0122,  0.0035,  0.0088,  0.0095,  0.0040,  0.0070,  0.0037,  0.0096,  0.0117,  0.0060,  0.0057, -0.1092,  0.0130],
        [ 0.0046,  0.0076,  0.0065,  0.0013,  0.0029,  0.0031,  0.0022,  0.0109,  0.0094,  0.0233,  0.0060,  0.0102,  0.0047,  0.0012,  0.0049, -0.0990],
        [ 0.0129,  0.0029,  0.0064,  0.0072,  0.0103,  0.0028,  0.0109,  0.0

In [108]:
dW_hand = G.T @ H[0]

dW_hand

tensor([[-0.0105,  0.0257, -0.0084, -0.0068],
        [ 0.0139, -0.0180,  0.0018,  0.0023],
        [ 0.0518, -0.1436, -0.0696,  0.1614],
        [-0.0209,  0.0202,  0.0174, -0.0166],
        [-0.0143,  0.0071, -0.0031,  0.0103],
        [-0.0107, -0.0124,  0.0209,  0.0021],
        [-0.0044,  0.0693, -0.0087, -0.0562],
        [ 0.0315, -0.0102, -0.0128, -0.0085],
        [ 0.0115, -0.0345,  0.0137,  0.0093],
        [ 0.1334, -0.1887,  0.1176, -0.0623],
        [ 0.0983,  0.1095, -0.1042, -0.1036],
        [-0.1642, -0.0473,  0.0652,  0.1462],
        [ 0.0569, -0.1502,  0.0988, -0.0055],
        [-0.1393,  0.2226,  0.0029, -0.0862],
        [ 0.0698,  0.0400, -0.2028,  0.0931],
        [-0.1029,  0.1107,  0.0713, -0.0791]], device='cuda:0', grad_fn=<MmBackward0>)

In [109]:
dW

tensor([[-0.0105,  0.0257, -0.0084, -0.0068],
        [ 0.0139, -0.0180,  0.0018,  0.0023],
        [ 0.0518, -0.1436, -0.0696,  0.1614],
        [-0.0209,  0.0202,  0.0174, -0.0166],
        [-0.0143,  0.0071, -0.0031,  0.0103],
        [-0.0107, -0.0124,  0.0209,  0.0021],
        [-0.0044,  0.0693, -0.0087, -0.0562],
        [ 0.0315, -0.0102, -0.0128, -0.0085],
        [ 0.0115, -0.0345,  0.0137,  0.0093],
        [ 0.1334, -0.1887,  0.1176, -0.0623],
        [ 0.0983,  0.1095, -0.1042, -0.1036],
        [-0.1642, -0.0473,  0.0652,  0.1462],
        [ 0.0569, -0.1502,  0.0988, -0.0055],
        [-0.1393,  0.2226,  0.0029, -0.0862],
        [ 0.0698,  0.0400, -0.2028,  0.0931],
        [-0.1029,  0.1107,  0.0713, -0.0791]], device='cuda:0')

In [110]:
db_hand = G.sum(dim=0)

db_hand


tensor([ 0.0615,  0.0388, -0.0726,  0.0493,  0.0339,  0.0333,  0.0909,  0.0654,  0.0445, -0.0313, -0.0694, -0.0432, -0.0396, -0.0577, -0.0667, -0.0374], device='cuda:0', grad_fn=<SumBackward1>)

In [111]:
db

tensor([ 0.0615,  0.0388, -0.0726,  0.0493,  0.0339,  0.0333,  0.0909,  0.0654,  0.0445, -0.0313, -0.0694, -0.0432, -0.0396, -0.0577, -0.0667, -0.0374], device='cuda:0')

In [112]:


dW_hand = torch.as_tensor(
    dW_hand,
    device=last_linear.weight.device,
    dtype=last_linear.weight.dtype,
).detach().clone()

db_hand = torch.as_tensor(
    db_hand,
    device=last_linear.bias.device,
    dtype=last_linear.bias.dtype,
).detach().clone()


# PyTorch Linear weight는 [out_features, in_features]
if dW_hand.shape != last_linear.weight.shape:
    if (
        dW_hand.ndim == 2
        and dW_hand.T.shape == last_linear.weight.shape
    ):
        print("dW_hand를 PyTorch weight 방향에 맞게 전치합니다.")
        dW_hand = dW_hand.T.contiguous()
    else:
        raise RuntimeError(
            f"dW_hand shape {tuple(dW_hand.shape)}와 "
            f"weight shape {tuple(last_linear.weight.shape)}가 다릅니다."
        )

if db_hand.numel() != last_linear.bias.numel():
    raise RuntimeError(
        f"db_hand shape {tuple(db_hand.shape)}와 "
        f"bias shape {tuple(last_linear.bias.shape)}가 다릅니다."
    )

db_hand = db_hand.reshape_as(last_linear.bias)

assert torch.isfinite(dW_hand).all()
assert torch.isfinite(db_hand).all()


# ============================================================
# 3. Adam 하이퍼파라미터
# ============================================================
learning_rate = 1e-3
beta1 = 0.9
beta2 = 0.999
epsilon = 1e-8
step_number = 1


# ============================================================
# 4. 업데이트 전 파라미터 저장
# ============================================================
W_before_hand = W_before
b_before_hand = b_before


# ============================================================
# 5. Adam 상태 초기화
# 첫 번째 업데이트이므로 모두 0
# ============================================================
mW_previous_hand = torch.zeros_like(W_before_hand)
vW_previous_hand = torch.zeros_like(W_before_hand)

mb_previous_hand = torch.zeros_like(b_before_hand)
vb_previous_hand = torch.zeros_like(b_before_hand)


# ============================================================
# 6. 1차 모멘트
# ============================================================
mW_hand = (
    beta1 * mW_previous_hand
    + (1.0 - beta1) * dW_hand
)

mb_hand = (
    beta1 * mb_previous_hand
    + (1.0 - beta1) * db_hand
)


# ============================================================
# 7. 2차 모멘트
# ============================================================
vW_hand = (
    beta2 * vW_previous_hand
    + (1.0 - beta2) * dW_hand.square()
)

vb_hand = (
    beta2 * vb_previous_hand
    + (1.0 - beta2) * db_hand.square()
)


# ============================================================
# 8. Bias correction
# ============================================================
mW_hat_hand = mW_hand / (
    1.0 - beta1**step_number
)

vW_hat_hand = vW_hand / (
    1.0 - beta2**step_number
)

mb_hat_hand = mb_hand / (
    1.0 - beta1**step_number
)

vb_hat_hand = vb_hand / (
    1.0 - beta2**step_number
)


# ============================================================
# 9. Adam 업데이트 직접 계산
# ============================================================
W_after_hand = (
    W_before_hand
    - learning_rate
    * mW_hat_hand
    / (torch.sqrt(vW_hat_hand) + epsilon)
)

b_after_hand = (
    b_before_hand
    - learning_rate
    * mb_hat_hand
    / (torch.sqrt(vb_hat_hand) + epsilon)
)







delta_W_hand = W_after_hand - W_before_hand
delta_b_hand = b_after_hand - b_before_hand


# ============================================================
# 12. 결과 출력
# ============================================================
print("\n" + "=" * 80)
print("수동 계산 Gradient")
print("=" * 80)

print("\ndW_hand:")
print(dW_hand.cpu())

print("\ndb_hand:")
print(db_hand.cpu())

print("\ndW_hand shape:", tuple(dW_hand.shape))
print("db_hand shape:", tuple(db_hand.shape))


print("\n" + "=" * 80)
print("업데이트 전 파라미터")
print("=" * 80)

print("\nW_before_hand:")
print(W_before_hand.cpu())

print("\nb_before_hand:")
print(b_before_hand.cpu())


print("\n" + "=" * 80)
print("Adam 모멘트")
print("=" * 80)

print("\nmW_hand:")
print(mW_hand.cpu())

print("\nvW_hand:")
print(vW_hand.cpu())

print("\nmb_hand:")
print(mb_hand.cpu())

print("\nvb_hand:")
print(vb_hand.cpu())


print("\n" + "=" * 80)
print("파라미터 변화량")
print("=" * 80)

print("\ndelta_W_hand:")
print(delta_W_hand.cpu())

print("\ndelta_b_hand:")
print(delta_b_hand.cpu())


print("\n" + "=" * 80)
print("수동 Adam 업데이트 후 파라미터")
print("=" * 80)

print("\nW_after_hand:")
print(W_after_hand.cpu())

print("\nb_after_hand:")
print(b_after_hand.cpu())






수동 계산 Gradient

dW_hand:
tensor([[-0.0105,  0.0257, -0.0084, -0.0068],
        [ 0.0139, -0.0180,  0.0018,  0.0023],
        [ 0.0518, -0.1436, -0.0696,  0.1614],
        [-0.0209,  0.0202,  0.0174, -0.0166],
        [-0.0143,  0.0071, -0.0031,  0.0103],
        [-0.0107, -0.0124,  0.0209,  0.0021],
        [-0.0044,  0.0693, -0.0087, -0.0562],
        [ 0.0315, -0.0102, -0.0128, -0.0085],
        [ 0.0115, -0.0345,  0.0137,  0.0093],
        [ 0.1334, -0.1887,  0.1176, -0.0623],
        [ 0.0983,  0.1095, -0.1042, -0.1036],
        [-0.1642, -0.0473,  0.0652,  0.1462],
        [ 0.0569, -0.1502,  0.0988, -0.0055],
        [-0.1393,  0.2226,  0.0029, -0.0862],
        [ 0.0698,  0.0400, -0.2028,  0.0931],
        [-0.1029,  0.1107,  0.0713, -0.0791]])

db_hand:
tensor([ 0.0615,  0.0388, -0.0726,  0.0493,  0.0339,  0.0333,  0.0909,  0.0654,  0.0445, -0.0313, -0.0694, -0.0432, -0.0396, -0.0577, -0.0667, -0.0374])

dW_hand shape: (16, 4)
db_hand shape: (16,)

업데이트 전 파라미터

W_before_hand:


In [113]:
dW

tensor([[-0.0105,  0.0257, -0.0084, -0.0068],
        [ 0.0139, -0.0180,  0.0018,  0.0023],
        [ 0.0518, -0.1436, -0.0696,  0.1614],
        [-0.0209,  0.0202,  0.0174, -0.0166],
        [-0.0143,  0.0071, -0.0031,  0.0103],
        [-0.0107, -0.0124,  0.0209,  0.0021],
        [-0.0044,  0.0693, -0.0087, -0.0562],
        [ 0.0315, -0.0102, -0.0128, -0.0085],
        [ 0.0115, -0.0345,  0.0137,  0.0093],
        [ 0.1334, -0.1887,  0.1176, -0.0623],
        [ 0.0983,  0.1095, -0.1042, -0.1036],
        [-0.1642, -0.0473,  0.0652,  0.1462],
        [ 0.0569, -0.1502,  0.0988, -0.0055],
        [-0.1393,  0.2226,  0.0029, -0.0862],
        [ 0.0698,  0.0400, -0.2028,  0.0931],
        [-0.1029,  0.1107,  0.0713, -0.0791]], device='cuda:0')

In [114]:
dW_hand

tensor([[-0.0105,  0.0257, -0.0084, -0.0068],
        [ 0.0139, -0.0180,  0.0018,  0.0023],
        [ 0.0518, -0.1436, -0.0696,  0.1614],
        [-0.0209,  0.0202,  0.0174, -0.0166],
        [-0.0143,  0.0071, -0.0031,  0.0103],
        [-0.0107, -0.0124,  0.0209,  0.0021],
        [-0.0044,  0.0693, -0.0087, -0.0562],
        [ 0.0315, -0.0102, -0.0128, -0.0085],
        [ 0.0115, -0.0345,  0.0137,  0.0093],
        [ 0.1334, -0.1887,  0.1176, -0.0623],
        [ 0.0983,  0.1095, -0.1042, -0.1036],
        [-0.1642, -0.0473,  0.0652,  0.1462],
        [ 0.0569, -0.1502,  0.0988, -0.0055],
        [-0.1393,  0.2226,  0.0029, -0.0862],
        [ 0.0698,  0.0400, -0.2028,  0.0931],
        [-0.1029,  0.1107,  0.0713, -0.0791]], device='cuda:0')

In [115]:
db

tensor([ 0.0615,  0.0388, -0.0726,  0.0493,  0.0339,  0.0333,  0.0909,  0.0654,  0.0445, -0.0313, -0.0694, -0.0432, -0.0396, -0.0577, -0.0667, -0.0374], device='cuda:0')

In [116]:
db_hand

tensor([ 0.0615,  0.0388, -0.0726,  0.0493,  0.0339,  0.0333,  0.0909,  0.0654,  0.0445, -0.0313, -0.0694, -0.0432, -0.0396, -0.0577, -0.0667, -0.0374], device='cuda:0')

In [119]:
W_after

tensor([[-0.4131, -0.0521, -0.4307, -0.1617],
        [ 0.3074, -0.0909,  0.0167,  0.1893],
        [ 0.2226,  0.1345, -0.0922,  0.3488],
        [-0.2684,  0.4611,  0.3898,  0.1903],
        [-0.4773,  0.1043, -0.1053,  0.4845],
        [-0.4374, -0.2418,  0.1929,  0.1119],
        [ 0.0416,  0.4786, -0.1482, -0.3480],
        [ 0.3437, -0.0654, -0.3527, -0.0322],
        [ 0.1226, -0.4009,  0.1493,  0.1344],
        [ 0.1921, -0.3718, -0.4036,  0.2745],
        [ 0.0475, -0.2120,  0.2201,  0.0234],
        [ 0.2575,  0.1583,  0.3743,  0.4551],
        [ 0.0527,  0.1694, -0.3881, -0.2697],
        [-0.1852,  0.4844, -0.1675, -0.1555],
        [ 0.0936,  0.2556, -0.2046,  0.2630],
        [-0.0220, -0.2763,  0.3830,  0.4876]], device='cuda:0')

In [120]:
W_after_hand

tensor([[-0.4131, -0.0521, -0.4307, -0.1617],
        [ 0.3074, -0.0909,  0.0167,  0.1893],
        [ 0.2226,  0.1345, -0.0922,  0.3488],
        [-0.2684,  0.4611,  0.3898,  0.1903],
        [-0.4773,  0.1043, -0.1053,  0.4845],
        [-0.4374, -0.2418,  0.1929,  0.1119],
        [ 0.0416,  0.4786, -0.1482, -0.3480],
        [ 0.3437, -0.0654, -0.3527, -0.0322],
        [ 0.1226, -0.4009,  0.1493,  0.1344],
        [ 0.1921, -0.3718, -0.4036,  0.2745],
        [ 0.0475, -0.2120,  0.2201,  0.0234],
        [ 0.2575,  0.1583,  0.3743,  0.4551],
        [ 0.0527,  0.1694, -0.3881, -0.2697],
        [-0.1852,  0.4844, -0.1675, -0.1555],
        [ 0.0936,  0.2556, -0.2046,  0.2630],
        [-0.0220, -0.2763,  0.3830,  0.4876]], device='cuda:0')

In [121]:
b_after

tensor([ 0.3082, -0.1327, -0.0415, -0.0494, -0.2922, -0.3502,  0.3807,  0.2897, -0.1482,  0.3918, -0.0766,  0.4216,  0.3659, -0.0261,  0.0563,  0.2578], device='cuda:0')

In [122]:
b_after_hand

tensor([ 0.3082, -0.1327, -0.0415, -0.0494, -0.2922, -0.3502,  0.3807,  0.2897, -0.1482,  0.3918, -0.0766,  0.4216,  0.3659, -0.0261,  0.0563,  0.2578], device='cuda:0')